## ECCO-TCP Fuzzy ECCOTCP Matching

### Overview¶

This notebook uses fuzzy matching to identify corresponding ECCO and ECCO-TCP documents.

In [ ]:
%pip install rapidfuzz unidecode

### 1. Import relevant libraries

In [ ]:
import sys
import csv
import pandas as pd
import numpy as np
import re
import unicodedata
import difflib

### 2. Define basic functions (no external dependencies)

In [ ]:
def unidecode(s):
    if s is None:
        return ""
    return unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode("ascii")

def extract_year(value):
    """
    Extract a plausible 4-digit year (1500–2029) from a string or number.
    Returns np.nan if no year is found.
    """
    if value is None or value == "" or pd.isna(value):
        return np.nan
    s = str(value)
    m = re.search(r"(1[5-9]\d{2}|20[0-2]\d)", s)  # 1500–2029
    if m:
        return int(m.group(0))
    return np.nan

def normalize_author(s):
    """
    Normalize author names for fuzzy matching:
    - lowercase, remove diacritics
    - drop bracketed info and life dates
    - keep alphabetic characters and spaces
    """
    if s is None or s == "" or pd.isna(s):
        return ""
    s = unidecode(str(s)).lower()

    # Drop anything after slash or bracketed notes
    s = s.split("/")[-1]
    s = re.split(r"\[|\(", s)[0]

    # Remove birth/death years patterns like "1667?-1723."
    s = re.sub(r"\d{3,4}\??-\d{2,4}\??\.?", " ", s)

    # Remove non-letters
    s = re.sub(r"[^a-z\s]", " ", s)

    # Collapse whitespace
    s = re.sub(r"\s+", " ", s).strip()
    return s

def normalize_title(s):
    """
    Normalize titles for fuzzy matching.

    Strategy:
    - lowercase, remove diacritics
    - keep only the main title segment: text before the first ':' or '.'
      (whichever comes first).
    - remove non-alphanumeric chars and collapse spaces
    """
    if s is None or s == "" or pd.isna(s):
        return ""
    s = unidecode(str(s)).lower()

    # Keep only main title portion (before first ':' or '.')
    # e.g.:
    #  "The basset-table: A comedy. ..." -> "the basset-table"
    #  "The basset-Table. A comedy. ..." -> "the basset-table"
    #  "The memoirs of an English officer: who served ..." -> "the memoirs of an english officer"
    s = re.split(r"[:\.]", s)[0]

    # Remove non-alphanumeric
    s = re.sub(r"[^a-z0-9\s]", " ", s)

    # Collapse whitespace
    s = re.sub(r"\s+", " ", s).strip()
    return s

def text_similarity(a, b):
    """
    Approximate similarity 0–100 using difflib.SequenceMatcher.
    """
    if not a or not b:
        return 0.0
    return 100.0 * difflib.SequenceMatcher(None, a, b).ratio()

def year_similarity(y1, y2):
    """
    Turn two years into a similarity score (0–100).
    Unknown year -> middling score (50).
    """
    if pd.isna(y1) or pd.isna(y2):
        return 50.0
    diff = abs(int(y1) - int(y2))
    # Simple decay: every year difference costs 2 points, up to 50 years
    return max(0.0, 100.0 - min(diff, 50) * 2.0)

def page_similarity(p1, p2):
    """
    Page-count similarity 0–100 based on relative difference.
    Unknown pages -> middling score (50).
    """
    if pd.isna(p1) or pd.isna(p2):
        return 50.0
    diff = abs(float(p1) - float(p2))
    denom = max(float(p1), float(p2))
    if denom == 0:
        return 50.0
    rel = diff / denom
    # Perfect match -> 100; if rel>=1, score -> 0
    return max(0.0, 100.0 * (1.0 - rel))

### 3. Load data

In [ ]:
TCP_PATH = "ECCOTCP_vols.csv"            #  TCP-as-volumes file
ECCO_META_PATH = "ecco_metadata_enriched.csv" # Full ECCO metadata

TCP_SEP = ","        # change to "\t" if tab-separated
ECCO_META_SEP = ","  # change to "\t" if tab-separated

tcp = pd.read_csv(TCP_PATH, sep=TCP_SEP, dtype=str, keep_default_na=False)
ecco_meta = pd.read_csv(ECCO_META_PATH, sep=ECCO_META_SEP, dtype=str, keep_default_na=False)

required_tcp_cols = {"TCP", "Author", "Date", "Title", "volume_number", "total_volumes"}
required_meta_cols = {
    "document_id", "title", "publication_year", "author",
    "volume_number", "total_volumes"
}

missing_tcp = required_tcp_cols - set(tcp.columns)
missing_meta = required_meta_cols - set(ecco_meta.columns)

if missing_tcp:
    raise ValueError(f"Missing ECCO-TCP columns: {missing_tcp}")
if missing_meta:
    raise ValueError(f"Missing ECCO metadata columns: {missing_meta}")

# Ensure optional page columns exist
if "Pages" not in tcp.columns:
    tcp["Pages"] = np.nan
if "total_pages" not in ecco_meta.columns:
    ecco_meta["total_pages"] = np.nan

### 4. Normalize and extract years, pages, volumes

In [ ]:
# ECCO-TCP with volumes
tcp["author_norm"] = tcp["Author"].apply(normalize_author)
tcp["title_norm"]  = tcp["Title"].apply(normalize_title)
tcp["year"]        = tcp["Date"].apply(extract_year).astype("float")
tcp["pages"]       = pd.to_numeric(tcp["Pages"], errors="coerce").astype("float")

tcp["vol_num"] = pd.to_numeric(tcp["volume_number"], errors="coerce").astype("float")
tcp["tot_vols"] = pd.to_numeric(tcp["total_volumes"], errors="coerce").astype("float")

# Full ECCO metadata with volumes
ecco_meta["author_norm"] = ecco_meta["author"].apply(normalize_author)
ecco_meta["title_norm"]  = ecco_meta["title"].apply(normalize_title)
ecco_meta["year"]        = ecco_meta["publication_year"].apply(extract_year).astype("float")
ecco_meta["pages"]       = pd.to_numeric(ecco_meta.get("total_pages", np.nan), errors="coerce").astype("float")

ecco_meta["vol_num"]  = pd.to_numeric(ecco_meta["volume_number"], errors="coerce").astype("float")
ecco_meta["tot_vols"] = pd.to_numeric(ecco_meta["total_volumes"], errors="coerce").astype("float")

### 5. Scoring candidates for ONE TCP record
    (using volume_number + total_volumes as ground-truth filter, plus new weights and a year gate)

In [ ]:
def score_candidates_for_tcp_record(
    tcp_row,
    meta_df,
    year_tolerance=2,
    page_tolerance=0.25,
    min_author_len=3,
    weight_author=0.35,
    weight_title=0.45,
    weight_year=0.15,
    weight_pages=0.05,
):
    """
    Given a single ECCO-TCP row and the full ECCO metadata DF, return
    a dataframe of candidate matches with similarity scores, sorted
    by match_score (descending).

    Volume filtering:
      - First restrict candidates to exact matches on vol_num and tot_vols
        (if available).
      - If no volume match is found, fall back to using the full metadata DF.
      - Volume fields are NOT included in the match_score; they are used
        only to select the candidate pool.

    Year gate:
      - If both years are known and year_score < 80, this is treated as
        almost certainly wrong and a match_score = 0 is forced.
    """
    t_author = tcp_row["author_norm"]
    t_title  = tcp_row["title_norm"]
    t_year   = tcp_row["year"]
    t_pages  = tcp_row["pages"]
    t_vnum   = tcp_row["vol_num"]
    t_tvols  = tcp_row["tot_vols"]

    candidates = meta_df
    volume_match_found = False

    # 1) Ground-truth filter by volume_number / total_volumes
    if not pd.isna(t_vnum) or not pd.isna(t_tvols):
        vol_mask = pd.Series(True, index=meta_df.index)
        if not pd.isna(t_vnum):
            vol_mask &= (meta_df["vol_num"] == t_vnum)
        if not pd.isna(t_tvols):
            vol_mask &= (meta_df["tot_vols"] == t_tvols)

        vol_subset = meta_df[vol_mask]
        if not vol_subset.empty:
            candidates = vol_subset
            volume_match_found = True
        # else: keep candidates as the full meta_df (fallback)

    # 2) Restrict by year window if TCP year is known (within chosen candidate pool)
    if not pd.isna(t_year):
        subset = candidates[
            candidates["year"].isna()
            | candidates["year"].between(t_year - year_tolerance, t_year + year_tolerance)
        ]
        if not subset.empty:
            candidates = subset

    # 3) Restrict by page band if TCP pages known
    if not pd.isna(t_pages):
        low = t_pages * (1.0 - page_tolerance)
        high = t_pages * (1.0 + page_tolerance)
        subset = candidates[
            candidates["pages"].isna()
            | candidates["pages"].between(low, high)
        ]
        if not subset.empty:
            candidates = subset

    # 4) Restrict by surname (first token of normalized author)
    surname = t_author.split(" ")[0] if t_author else ""
    if surname and len(surname) >= min_author_len:
        mask = candidates["author_norm"].str.contains(
            rf"\b{re.escape(surname)}\b", case=False, na=False
        )
        if mask.any():
            candidates = candidates[mask]

    rows = []
    for _, m_row in candidates.iterrows():
        a_score = text_similarity(t_author, m_row["author_norm"])
        ti_score = text_similarity(t_title, m_row["title_norm"])
        y_score = year_similarity(t_year, m_row["year"])
        p_score = page_similarity(t_pages, m_row["pages"])
    
        # Year gate: only apply when BOTH years are known
        has_both_years = not (pd.isna(t_year) or pd.isna(m_row["year"]))
        if has_both_years and y_score < 80:
            overall = 0.0
        else:
            overall = (
                weight_author * a_score
                + weight_title * ti_score
                + weight_year * y_score
                + weight_pages * p_score
            )

        rows.append(
            {
                "TCP": tcp_row["TCP"],
                "document_id": m_row["document_id"],
                "estc_id": m_row.get("estc_id", np.nan),
                "tcp_author": tcp_row["Author"],
                "ecco_author": m_row["author"],
                "tcp_title": tcp_row["Title"],
                "ecco_title": m_row["title"],
                "tcp_year": t_year,
                "ecco_year": m_row["year"],
                "tcp_pages": t_pages,
                "ecco_pages": m_row["pages"],
                "tcp_volume_number": t_vnum,
                "tcp_total_volumes": t_tvols,
                "ecco_volume_number": m_row["vol_num"],
                "ecco_total_volumes": m_row["tot_vols"],
                "author_score": a_score,
                "title_score": ti_score,
                "year_score": y_score,
                "pages_score": p_score,
                "match_score": overall,
                "volume_match_found": volume_match_found,
            }
        )

    if not rows:
        return pd.DataFrame()

    result = pd.DataFrame(rows).sort_values("match_score", ascending=False)
    return result

### 6. Best single match per TCP record (one row per TCP)

In [ ]:
def match_best_tcp_to_ecco_meta(
    tcp_df,
    meta_df,
    year_tolerance=2,
    page_tolerance=0.25,
):
    """
    For each ECCO-TCP record, find the single best ECCO metadata candidate.
    Returns a DataFrame with exactly len(tcp_df) rows.
    """
    results = []
    n = len(tcp_df)

    for i, (_, t_row) in enumerate(tcp_df.iterrows(), start=1):
        candidates = score_candidates_for_tcp_record(
            t_row,
            meta_df,
            year_tolerance=year_tolerance,
            page_tolerance=page_tolerance,
        )

        if not candidates.empty:
            best = candidates.iloc[0].to_dict()
        else:
            # No plausible candidate found: keep TCP info, mark ECCO as missing
            best = {
                "TCP": t_row["TCP"],
                "document_id": np.nan,
                "estc_id": np.nan,
                "tcp_author": t_row["Author"],
                "ecco_author": np.nan,
                "tcp_title": t_row["Title"],
                "ecco_title": np.nan,
                "tcp_year": t_row["year"],
                "ecco_year": np.nan,
                "tcp_pages": t_row["pages"],
                "ecco_pages": np.nan,
                "tcp_volume_number": t_row["vol_num"],
                "tcp_total_volumes": t_row["tot_vols"],
                "ecco_volume_number": np.nan,
                "ecco_total_volumes": np.nan,
                "author_score": np.nan,
                "title_score": np.nan,
                "year_score": np.nan,
                "pages_score": np.nan,
                "match_score": 0.0,
                "volume_match_found": False,
            }

        results.append(best)

        # Optional simple progress indicator
        if i % 200 == 0 or i == n:
            print(f"Processed {i}/{n} TCP records")

    return pd.DataFrame(results)

### 7. Run matching algorithm

In [ ]:
best_tcp_matches = match_best_tcp_to_ecco_meta(
    tcp,
    ecco_meta,
    year_tolerance=2,    # tighten/loosen if needed
    page_tolerance=0.25  # 25% band around TCP page count
)

print("Number of ECCO-TCP records:", len(tcp))
print("Number of rows in best_tcp_matches:", len(best_tcp_matches))

# Peek at results
display(best_tcp_matches.head())

### 8. Clean numeric columns and save to CSV


In [ ]:
numeric_cols = [
    "tcp_year", "ecco_year",
    "tcp_pages", "ecco_pages",
    "tcp_volume_number", "tcp_total_volumes",
    "ecco_volume_number", "ecco_total_volumes",
    "author_score", "title_score",
    "year_score", "pages_score",
    "match_score",
]

for col in numeric_cols:
    if col in best_tcp_matches.columns:
        best_tcp_matches[col] = pd.to_numeric(best_tcp_matches[col], errors="coerce")

# Years, pages, volumes as whole numbers
for col in [
    "tcp_year", "ecco_year",
    "tcp_pages", "ecco_pages",
    "tcp_volume_number", "tcp_total_volumes",
    "ecco_volume_number", "ecco_total_volumes",
]:
    if col in best_tcp_matches.columns:
        best_tcp_matches[col] = best_tcp_matches[col].round()

# Scores with 3 decimal places
score_cols = ["author_score", "title_score", "year_score", "pages_score", "match_score"]
for col in score_cols:
    if col in best_tcp_matches.columns:
        best_tcp_matches[col] = best_tcp_matches[col].round(3)

OUTPUT_PATH = "ECCOTCP_vols_to_full_ECCO_best_match_clean.csv"
best_tcp_matches.to_csv(OUTPUT_PATH, index=False, float_format="%.3f")
print(f"Saved cleaned matches to: {OUTPUT_PATH}")